# 생성기·길이·MP3 조건을 맞춘 비교
T4 GPU에서 모두 실행하고 generator_check_bundle.zip을 선택하세요.
기존 음원 84개를 두 길이·두 코덱 조건으로 처리하고 SONICS와 DF-Arena를 비교합니다. 과거 holdout도 이미 사용한 개발 자료로 취급합니다. 학습·대회 제출은 하지 않습니다.
DF-Arena 가중치 다운로드에 약 4.6GB가 필요할 수 있습니다. 결과: generator_results.zip


In [ ]:
from google.colab import files
from pathlib import Path
import torch, sys, json, hashlib, zipfile, io, tempfile, shutil, subprocess, os
assert torch.cuda.is_available(), "런타임 유형을 T4 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0), torch.__version__, sys.version)
uploaded=files.upload()
assert len(uploaded)==1, "generator_check_bundle.zip 하나만 선택하세요."
blob=next(iter(uploaded.values()))
assert hashlib.sha256(blob).hexdigest()=="cefacd910f4aa6ff8e13c932651d18235b92c841aa6977f758b5829173ae136e", "ZIP 버전이 다릅니다."
WORK=Path(tempfile.mkdtemp(prefix="generator_check_",dir="/content"))
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for name in z.namelist():
        assert (WORK/name).resolve().is_relative_to(WORK.resolve())
    z.extractall(WORK)
plan=json.loads((WORK/"plan.json").read_text())
for name,sha in plan["file_sha256"].items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest()==sha, name
RESULTS=WORK/"generator_results"
RESULTS.mkdir()
print("84개 입력과 실행 코드 무결성 확인 완료")


In [ ]:
# torch/torchvision을 임의로 업그레이드하지 않는다.
installed=subprocess.run([sys.executable,"-m","pip","install","--no-deps","timm==1.0.15"],capture_output=True,text=True)
(RESULTS/"install.log").write_text(installed.stdout+installed.stderr)
print((installed.stdout+installed.stderr)[-4000:])
installed.check_returncode()
extra=subprocess.run([sys.executable,"-m","pip","install","demucs==4.0.1"],capture_output=True,text=True)
with (RESULTS/"install.log").open("a") as log:
    log.write(extra.stdout+extra.stderr)
print((extra.stdout+extra.stderr)[-2000:])
extra.check_returncode()
check=subprocess.run([sys.executable,"-c","import timm, torch, torchvision, torchaudio, librosa, soundfile, huggingface_hub; from sonics import HFAudioClassifier; import script; print('의존성 검사 통과', timm.__version__)"],cwd=WORK,capture_output=True,text=True)
(RESULTS/"preflight.log").write_text(check.stdout+check.stderr)
print(check.stdout+check.stderr)
check.check_returncode()
assert shutil.which("ffmpeg"), "ffmpeg가 없습니다."
(RESULTS/"environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)+"\n"+sys.version)


In [ ]:
from huggingface_hub import hf_hub_download
for variant,entry in plan["models"].items():
    cached=Path(hf_hub_download(repo_id=entry["repo"],filename="pytorch_model.bin",revision=entry["revision"]))
    assert cached.stat().st_size==entry["weights_size"]
    assert hashlib.sha256(cached.read_bytes()).hexdigest()==entry["weights_sha256"], variant
    shutil.copy2(cached,WORK/"weights"/variant/"pytorch_model.bin")
    print(variant,"가중치 해시 검사 통과")

entry=plan["df_model"]
cached=Path(hf_hub_download(repo_id=entry["repo"],filename="pytorch_model.bin",revision=entry["revision"]))
h=hashlib.sha256()
with cached.open("rb") as stream:
    for block in iter(lambda:stream.read(4*1024**2),b""):
        h.update(block)
assert h.hexdigest()==plan["weight_sha256"]["df_arena"]
target=WORK/"model/df_arena_1b/pytorch_model.bin"
try:
    os.link(cached,target)
except OSError:
    shutil.copy2(cached,target)
print("DF-Arena 가중치 해시 검사 통과")


In [ ]:
# 추론은 로컬 가중치를 사용하며 HF 네트워크 다운로드를 금지한다.
env=os.environ.copy()
env["HF_HUB_OFFLINE"]="1"
env["TRANSFORMERS_OFFLINE"]="1"
try:
    with (RESULTS/"run.log").open("w",encoding="utf-8") as log:
        process=subprocess.Popen([sys.executable,"-u","run_generator_check.py","--work",str(WORK)],cwd=WORK,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end="")
                log.write(line)
                log.flush()
            code=process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
    assert code==0, "검증이 중단됐습니다. 결과 ZIP과 오류 내용을 보내주세요."
finally:
    archive=shutil.make_archive("/content/generator_results","zip",RESULTS)
    files.download(archive)
